# 03 展示：生成仪表盘与参数实验室

从数据库读 `02_回测.ipynb` 写入的结果，生成四个网页，直接写进 `docs/`（GitHub Pages）：

| 页面 | 内容 |
|---|---|
| `dashboard.html` / `dashboard_en.html` | 回测仪表盘（中文 / 英文），五个指数的完整结果 |
| `lab.html` / `lab_en.html` | 参数实验室，拖滑块在浏览器里实时重算轻参数 |

页面上的文字都在 `assets/i18n.js` 的中英对照表里；图表引擎、回测逻辑的 JS 镜像和样式也在 `assets/`。
这些是网页本身需要的前端文件，所以留在仓库里。

想用 Tableau 做图：直接连 PostgreSQL 数据库 `hindenburg`，用表 `daily`、`clusters`、`hit_vs_base`。

## 0. 设置

In [1]:
import json
import os
from pathlib import Path

import pandas as pd
import sqlalchemy as sa

DB_URL = os.getenv('HO_DB_URL', 'postgresql+psycopg://localhost/hindenburg')
OUT_DIR = Path(os.getenv('HO_PAGES_DIR', 'docs'))     # 网页输出目录
ASSETS = Path('assets')
LANGS = ('zh', 'en')                                  # 生成的语言版本
engine = sa.create_engine(DB_URL)

# 仪表盘里策略的顺序（实验对照组不进仪表盘，只进参数实验室）
DASHBOARD_ORDER = ['ashare', 'general', '300', '500', '1000']

# 参数实验室的滑块（取值范围；名称和单位的中英文在 assets/i18n.js 的 sl_* / u_*）
SLIDER_SPECS = [
    dict(key='ma_period', min=10, max=120, step=5),
    dict(key='ma_lookback', min=1, max=20, step=1),
    dict(key='threshold', min=0.005, max=0.05, step=0.001, pct=True),
    dict(key='hl_ratio', min=0.5, max=5.0, step=0.1),
    dict(key='cluster_window', min=5, max=60, step=1),
    dict(key='cluster_min', min=1, max=6, step=1),
]

## 1. 从数据库组装页面数据

每个策略一份数据：参数、汇总统计、逐日信号标注、回测起点之前的指数（供浏览器算均线）、
中间序列（参数实验室靠它们重算轻参数）、集群列表。

In [2]:
def query(sql: str, **params) -> pd.DataFrame:
    with engine.connect() as con:
        return pd.read_sql(sa.text(sql), con, params=params)


def payload(key: str) -> dict:
    with engine.connect() as con:
        info = con.execute(sa.text('SELECT * FROM strategy_info WHERE strategy = :k'), {'k': key}).mappings().one()
        meta = con.execute(sa.text('SELECT meta FROM summary WHERE strategy = :k'), {'k': key}).scalar()
    params = info['params']
    df = query('SELECT * FROM daily WHERE strategy = :k ORDER BY trade_date', k=key)
    pre = query('SELECT trade_date, close FROM index_pre WHERE strategy = :k ORDER BY trade_date', k=key)
    cl = query('SELECT * FROM clusters WHERE strategy = :k ORDER BY cluster_id', k=key)

    def num(v, nd=None, as_int=False):
        if v is None or pd.isna(v):
            return None
        return int(v) if as_int else (round(float(v), nd) if nd is not None else float(v))

    def col(name, nd=None, as_int=False):
        return [num(v, nd, as_int) for v in df[name]]

    sig = df.index[df['signal']]
    return {
        'strategy': info['name'], 'benchmark': info['benchmark'], 'index_source': info['index_source'],
        'state_rule': info['state_rule'], 'params': params, 'heavy_keys': info['heavy_keys'], 'meta': meta,
        'signal_days': [{'date': str(df.at[i, 'trade_date']), 'index': int(i), 'level': df.at[i, 'signal_level'],
                         'cluster_id': num(df.at[i, 'cluster_id'], as_int=True)} for i in sig],
        'index_pre': {'dates': [str(d) for d in pre['trade_date']], 'index_close': [num(v, 3) for v in pre['close']]},
        'series': {
            'dates': [str(d) for d in df['trade_date']],
            'index_close': col('index_close', 3), 'high_ratio': col('high_ratio', 5), 'low_ratio': col('low_ratio', 5),
            'new_high': col('new_high', as_int=True), 'new_low': col('new_low', as_int=True),
            'mcclellan': col('mcclellan', 4)},
        'clusters': [
            {'id': int(r.cluster_id), 'start': str(r.start_date), 'end': str(r.end_date), 'count': int(r.signal_count),
             'confirm': None if pd.isna(r.confirm_date) else str(r.confirm_date), 'anchor': str(r.anchor_date),
             'type': r.type, 'level': r.level,
             **{f'dd{w}': num(getattr(r, f'dd_{w}d'), 4) for w in params['dd_windows']}}
            for r in cl.itertuples()],
    }


keys = query('SELECT strategy, experimental FROM strategy_info')
PAYLOADS = {k: payload(k) for k in sorted(keys['strategy'])}
print('已读取：', ', '.join(f"{k}（{p['strategy']}）" for k, p in PAYLOADS.items()))

已读取： 1000（中证1000）, 300（沪深300）, 500（中证500）, ashare（中证全指）, ashare_unadj（中证全指(不复权+不过滤·实验对照)）, general（上证综指）


## 2. 页面模板

下面几格是网页的 HTML 模板和页面脚本。页面结构写在这里，文字从 `assets/i18n.js` 取，
所以同一份模板能生成中英两版。

In [3]:
# 回撤等级 → 颜色。键必须和 backtest.grade() 的返回值一致
LEVEL_COLOR = {'较大回撤': '#C64B37', '中型回调': '#E0904A', '小幅回调': '#D9A441',
               '无显著回撤': '#4FA98C', '近期信号': '#5C8AE6'}

# 网页标题（<title> 在 JS 运行前就要有，所以单独放在这里）
TITLES = {
    'dashboard': {'zh': '兴登堡预兆 · 回测仪表盘', 'en': 'Hindenburg Omen · Backtest Dashboard'},
    'lab': {'zh': '兴登堡预兆 · 参数实验室', 'en': 'Hindenburg Omen · Parameter Lab'},
}


def page_name(kind: str, lang: str) -> str:
    """页面文件名：dashboard.html / dashboard_en.html / lab.html / lab_en.html"""
    return f'{kind}.html' if lang == 'zh' else f'{kind}_{lang}.html'


def _pct(x, d=0):
    return '—' if x is None else f'{x*100:.{d}f}%'


def _asset(name: str) -> str:
    return (ASSETS / name).read_text(encoding='utf-8')


def toggle_buttons(payloads: dict) -> str:
    """策略切换按钮的 HTML（第一个默认选中；按钮文字由 i18n.js 按语言填充）。"""
    return '\n'.join(
        f'<button class="toggle-btn{" active" if i == 0 else ""}" data-cfg="{k}" type="button" '
        f'data-v="{payloads[k]["strategy"]}">{payloads[k]["strategy"]}</button>'
        for i, k in enumerate(payloads))


def lang_link(kind: str, lang: str) -> str:
    """导航栏右侧切换到另一种语言的链接。"""
    other = 'en' if lang == 'zh' else 'zh'
    return (f'<a class="lang-switch" href="{page_name(kind, other)}" '
            f'data-i18n="lang_switch"></a>')


def _head(title: str, extra_css: str = '', lang: str = 'zh') -> str:
    """HTML 头部：字体、内联 report.css 和额外样式。"""
    html_lang = 'zh-CN' if lang == 'zh' else 'en'
    return f"""<!DOCTYPE html>
<html lang="{html_lang}">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>{title}</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link href="https://fonts.googleapis.com/css2?family=Noto+Serif+SC:wght@500;700&family=Noto+Sans+SC:wght@400;500;600&family=IBM+Plex+Mono:wght@400;500;600&display=swap" rel="stylesheet">
<style>
{_asset('report.css')}
{extra_css}
.topnav .lang-switch{{margin-left:auto;}}
</style>
</head>
<body>"""


def _scripts(lang: str, data_js: str, page_js: str) -> str:
    """页面末尾的脚本：先定语言和对照表，再加载图表引擎、回测镜像、共用工具和页面逻辑。"""
    return f"""
<script>window.__LANG__ = {json.dumps(lang)};</script>
<script>{_asset('i18n.js')}</script>
<script>{_asset('chart_engine.js')}</script>
<script>{_asset('backtest.js')}</script>
<script>{_asset('common.js')}</script>
<script>
{data_js}
</script>
<script>
I18N.apply();
{page_js}
</script>
</body></html>"""

In [4]:
# 仪表盘专用样式（策略切换按钮、对比表）
DASHBOARD_CSS = """
.toggle-row{display:flex; align-items:center; gap:12px; margin:0 0 24px; flex-wrap:wrap;}
.toggle-group{display:inline-flex; background:var(--card); border:1px solid var(--line); border-radius:10px; padding:4px;}
.toggle-btn{font-family:var(--font-body); font-size:13.5px; font-weight:500; color:var(--text-dim);
  background:none; border:none; padding:8px 18px; border-radius:7px; cursor:pointer; transition:.15s;}
.toggle-btn.active{background:var(--amber); color:#FFFFFF; font-weight:600;}
.toggle-btn:not(.active):hover{color:var(--text);}
.toggle-caption{font-size:12.5px; color:var(--text-faint); font-family:var(--font-mono);}
.cmp-tbl th, .cmp-tbl td{text-align:center;}
.cmp-tbl th:first-child, .cmp-tbl td:first-child{text-align:left;}
"""


# =====================================================================
# 仪表盘
# =====================================================================

def build_dashboard(payloads: dict, out_path: Path, lang: str = 'zh') -> Path:
    """payloads: {策略键: export_payload() 的结果}，按 lang 写出仪表盘。"""
    keys = list(payloads)

    btns = toggle_buttons(payloads)

    # 策略对比：泳道图展示各策略有效集群的时间分布，±TOL 个交易日内重叠视为同一事件。
    # 日期轴必须用所有策略日期的并集（各策略起点不同），否则早于某策略起点的集群
    # 会找不到位置。
    cmp_section = ''
    if len(keys) >= 2:
        all_dates = sorted({d for k in keys for d in payloads[k]['series']['dates']})
        idx = {d: i for i, d in enumerate(all_dates)}
        TOL = 8

        def valid_of(k):
            return [c for c in payloads[k]['clusters'] if c['type'] == '有效集群']

        missing = [(k, c['start']) for k in keys for c in valid_of(k)
                   if c['start'] not in idx or c['end'] not in idx]
        if missing:
            print(f'  ⚠ 策略对比: {len(missing)} 个集群的日期不在并集轴上, 已跳过: {missing[:3]}')

        # 裁掉最早集群之前的空白，左侧留 60 个交易日
        first_i = min((idx[c['start']] for k in keys for c in valid_of(k)
                       if c['start'] in idx), default=0)
        clip = max(0, first_i - 60)
        all_dates = all_dates[clip:]
        idx = {d: i - clip for d, i in idx.items() if i >= clip}

        # 泳道数据只放原始字段，显示文字由页面 JS 按语言拼出
        lane_colors = ['#D9A441', '#5C8AE6', '#4FA98C', '#C64B37', '#E0904A']
        lanes, per_key = [], {}
        for li, k in enumerate(keys):
            pk = payloads[k]
            items = []
            for c in valid_of(k):
                if c['start'] not in idx or c['end'] not in idx:
                    continue
                items.append(dict(id=c['id'], i0=idx[c['start']], i1=idx[c['end']],
                                  color=LEVEL_COLOR.get(c['level'], '#5C8AE6'),
                                  strategy=pk['strategy'], level=c['level'],
                                  start=c['start'], end=c['end'], count=c['count']))
            per_key[k] = items
            lanes.append(dict(label=pk['strategy'],
                              color=lane_colors[li % len(lane_colors)], items=items))

        # 相邻泳道中时间重叠的集群连线
        links = []
        for li in range(len(keys) - 1):
            for ca in per_key[keys[li]]:
                for cb in per_key[keys[li + 1]]:
                    if not (ca['i1'] + TOL < cb['i0'] or cb['i1'] + TOL < ca['i0']):
                        links.append([li, ca['id'], li + 1, cb['id']])

        def _base_of(pk, field):
            ba = pk['meta'].get('baseline_all') or {}
            gw = pk['params']['grade_window']
            # JSON 往返后 int 键会变成字符串，两种都试
            return (ba.get(str(gw)) or ba.get(gw) or {}).get(field)

        head_cells = ''.join(f'<th data-v="{payloads[k]["strategy"]}"></th>' for k in keys)
        cmp_metrics = [
            ('cmp_period', lambda pk: f'{pk["meta"]["start"]} ~ {pk["meta"]["end"]}'),
            ('cmp_bench', lambda pk: f'<span data-v="{pk["index_source"]}"></span>'),
            ('cmp_nvalid', lambda pk: pk['meta']['n_clusters_valid']),
            ('cmp_hit_mid', lambda pk: _pct(pk['meta']['hit_rate'])),
            ('cmp_base_mid', lambda pk: _pct(_base_of(pk, 'p_mid'))),
            ('cmp_hit_big', lambda pk: _pct(pk['meta'].get('hit_rate_big'))),
            ('cmp_base_big', lambda pk: _pct(_base_of(pk, 'p_big'))),
        ]
        cmp_rows = '\n'.join(
            f'<tr><td class="strong" data-i18n="{key}"></td>'
            + ''.join(f'<td class="num">{fn(payloads[k])}</td>' for k in keys)
            + '</tr>' for key, fn in cmp_metrics)

        cmp_section = f"""
<section class="section" id="compare">
  <div class="section-head"><span class="section-num" data-i18n="num6"></span><h2 data-i18n="sec_compare"></h2></div>
  <p class="section-sub" data-i18n="compare_sub" data-arg="{TOL}"></p>
  <div id="chart-timeline"></div>
  <div style="height:16px"></div>
  <div class="tbl-wrap"><div class="tbl-scroll"><table class="rpt cmp-tbl">
    <thead><tr><th data-i18n="cmp_metric"></th>{head_cells}</tr></thead>
    <tbody>{cmp_rows}</tbody></table></div></div>
</section>
<script>window.__TIMELINE__ = {json.dumps(dict(dates=all_dates, lanes=lanes, links=links), ensure_ascii=False)};</script>
"""

    html = _head(TITLES['dashboard'][lang], DASHBOARD_CSS, lang) + f"""
<nav class="topnav">
  <span class="brand">HINDENBURG·DASHBOARD</span>
  <a href="#summary" data-i18n="nav_summary"></a><a href="#trend" data-i18n="nav_trend"></a>
  <a href="#panel" data-i18n="nav_panel"></a><a href="#history" data-i18n="nav_history"></a>
  <a href="#stats" data-i18n="nav_stats"></a>{'<a href="#compare" data-i18n="nav_compare"></a>' if cmp_section else ''}
  <a href="{page_name('lab', lang)}">LAB</a>
  {lang_link('dashboard', lang)}
</nav>

<header class="hero">
  <h1 data-i18n="hero_dash"></h1>
  <div class="meta-row">
    <span><b data-i18n="sample_period"></b><span data-i18n="colon"></span><span id="hero-range"></span></span>
  </div>
</header>

<div class="section" style="padding-bottom:0">
  <div class="toggle-row">
    <div class="toggle-group">{btns}</div>
    <span class="toggle-caption" id="toggle-caption"></span>
  </div>
</div>

<section class="section" id="status">
  <div class="section-head"><span class="section-num">·</span><h2 data-i18n="sec_status"></h2></div>
  <p class="section-sub" data-i18n="sec_status_sub"></p>
  <div class="stat-grid" id="status-grid"></div>
</section>

<section class="section" id="summary">
  <div class="section-head"><span class="section-num" data-i18n="num1"></span><h2 data-i18n="sec_summary"></h2></div>
  <div class="stat-grid" id="stat-grid"></div>
  <p class="section-sub" style="margin-bottom:10px" data-i18n="sec_summary_sub"></p>
  <div class="pill-row" id="pill-row"></div>
  <div class="section-note warn" id="summary-note"></div>
</section>

<section class="section" id="trend">
  <div class="section-head"><span class="section-num" data-i18n="num2"></span><h2 id="trend-title"></h2></div>
  <p class="section-sub" data-i18n="sec_trend_sub"></p>
  <div id="chart-trend"></div>
</section>

<section class="section" id="panel">
  <div class="section-head"><span class="section-num" data-i18n="num3"></span><h2 data-i18n="sec_panel"></h2></div>
  <p class="section-sub" id="panel-sub"></p>
  <div class="chart-grid cols-2-2">
    <div id="chart-cond1"></div><div id="chart-cond2"></div>
    <div id="chart-cond3"></div><div id="chart-cond4"></div>
  </div>
</section>

<section class="section" id="history">
  <div class="section-head"><span class="section-num" data-i18n="num4"></span><h2 data-i18n="sec_history"></h2></div>
  <p class="section-sub" id="history-sub"></p>
  <div class="tbl-wrap"><div class="tbl-scroll"><table class="rpt">
    <thead><tr><th data-i18n="col_cluster"></th><th data-i18n="col_period"></th><th data-i18n="col_count"></th><th data-i18n="col_type"></th><th id="history-ddhead"></th><th data-i18n="col_grade"></th></tr></thead>
    <tbody id="history-tbody"></tbody></table></div></div>
</section>

<section class="section" id="stats">
  <div class="section-head"><span class="section-num" data-i18n="num5"></span><h2 data-i18n="sec_stats"></h2></div>
  <p class="section-sub" id="stats-sub"></p>
  <div id="chart-box"></div>
</section>

{cmp_section}

<footer>
  <p id="foot"></p>
</footer>
""" + _scripts(lang, f"""window.__DATA__ = {json.dumps(payloads, ensure_ascii=False)};
window.__LEVEL_COLOR__ = {json.dumps(LEVEL_COLOR, ensure_ascii=False)};""", DASHBOARD_JS)

    out_path.write_text(html, encoding='utf-8')
    return out_path

In [5]:
DASHBOARD_JS = r"""
(function(){
  const D = window.__DATA__, LC = window.__LEVEL_COLOR__;
  const {TAG, pct} = window.HO;
  const {t, v} = window.I18N;
  let charts = [];


  function render(key){
    charts.forEach(c => c.destroy && c.destroy());
    charts = [];
    const d = D[key], S = d.series, m = d.meta, p = d.params;
    const src = v(d.index_source);

    document.getElementById('toggle-caption').textContent = t('caption', v(d.strategy), src, p.cluster_window);
    document.querySelectorAll('.toggle-btn').forEach(b =>
      b.classList.toggle('active', b.dataset.cfg === key));

    document.getElementById('hero-range').textContent = t('range', m.start, m.end, m.n_days.toLocaleString());

    // 口径说明按实际参数生成
    document.getElementById('foot').innerHTML = t('foot',
      (p.dd_mode === 'peak_trough' ? t('dd_peak') : t('dd_from')) + t('anchor', p.dd_anchor === 'confirm'));

    // 当前状态（现状快照，不是回测统计）
    const st = m.status || {};
    const ac = st.active_cluster;
    document.getElementById('status-grid').innerHTML = `
      <div class="stat-card"><div class="label">${t('st_last')}</div>
        <div class="value" style="font-size:20px">${st.last_signal_date || t('none')}</div>
        <div class="foot">${t('st_asof', st.as_of || '-')}</div></div>
      <div class="stat-card"><div class="label">${t('st_recent', st.window)}</div>
        <div class="value">${st.recent_signal_count ?? 0}</div>
        <div class="foot">${t('st_recent_foot')}</div></div>
      <div class="stat-card"><div class="label">${t('st_cluster')}</div>
        <div class="value" style="font-size:16px">${ac ? v(ac.type) : t('none')}</div>
        <div class="foot">${ac
          ? t('st_cluster_foot', ac.start, ac.end, ac.count) +
            (ac.confirm_date ? t('st_confirmed', ac.confirm_date) : t('st_unconfirmed'))
          : t('st_no_cluster', st.window)}</div></div>`;

    const gw = p.grade_window, cuts = p.grade_cuts;
    document.getElementById('stat-grid').innerHTML = `
      <div class="stat-card"><div class="label">${t('sc_total')}</div>
        <div class="value">${m.n_clusters_valid}</div>
        <div class="foot">${t('sc_total_foot', m.n_clusters_valid - m.n_gradable)}</div></div>
      <div class="stat-card"><div class="label">${t('sc_mid')}</div>
        <div class="value">${m.hit_mid}</div>
        <div class="foot">${t('sc_win', gw, (cuts[1]*100).toFixed(0))}</div></div>
      <div class="stat-card"><div class="label">${t('sc_big')}</div>
        <div class="value">${m.hit_big}</div>
        <div class="foot">${t('sc_win', gw, (cuts[0]*100).toFixed(0))}</div></div>
      <div class="stat-card"><div class="label">${t('sc_rate')}</div>
        <div class="value">${pct(m.hit_rate,0)}</div>
        <div class="foot">${t('sc_rate_foot', m.n_gradable)}</div></div>`;

    const order = ['较大回撤','中型回调','小幅回调','无显著回撤','近期信号'];
    document.getElementById('pill-row').innerHTML = order.filter(l => m.level_counts[l])
      .map(l => `<div class="pill"><span class="dot" style="background:${LC[l]}"></span>${v(l)} <b>${m.level_counts[l]}</b></div>`).join('');

    const base = (m.baseline_all || {})[gw];
    document.getElementById('summary-note').innerHTML =
      t('summary_note', m.n_gradable, pct(m.hit_rate,0), pct(base ? base.p_mid : null, 0));

    // 走势与逐日信号标注
    document.getElementById('trend-title').textContent = t('trend_title', src);
    const markers = HO.signalMarkers(d.signal_days, d.clusters, LC);
    charts.push(new CE.TimeSeriesChart(document.getElementById('chart-trend'), {
      title: src, subtitle: t('trend_sub'), dates:S.dates, height:340,
      series:[{label:src, color:'#1B2233', data:S.index_close, width:1.5}],
      markers, yFormat:v=>v.toFixed(0)}));

    const ma = BT.rollingMean(S.index_close, p.ma_period);
    charts.push(new CE.TimeSeriesChart(document.getElementById('chart-cond1'), {
      title:t('c1_title'), subtitle:t('c1_sub', src, p.ma_period), dates:S.dates, height:230,
      series:[{label:t('index'),color:'#97A3BC',data:S.index_close,width:1.1},
              {label:`MA${p.ma_period}`,color:'#D9A441',data:ma,width:1.8}],
      yFormat:v=>v.toFixed(0)}));
    charts.push(new CE.TimeSeriesChart(document.getElementById('chart-cond2'), {
      title:t('c2_title'), subtitle:t('c2_sub', (p.threshold*100).toFixed(1)), dates:S.dates, height:230,
      series:[{label:t('new_high_pct'),color:'#C64B37',data:S.high_ratio,width:1.3},
              {label:t('new_low_pct'),color:'#5C8AE6',data:S.low_ratio,width:1.3}],
      hlines:[{value:p.threshold,color:'#D9A441',label:t('threshold')}], yFormat:v=>(v*100).toFixed(1)+'%'}));
    charts.push(new CE.TimeSeriesChart(document.getElementById('chart-cond3'), {
      title:t('c3_title', p.hl_ratio), subtitle:t('c3_sub'), dates:S.dates, height:230,
      series:[{label:t('new_high_n'),color:'#C64B37',data:S.new_high,type:'bar',opacity:0.75},
              {label:t('threshold'),color:'#5C8AE6',data:S.new_low.map(v=>v==null?null:v*p.hl_ratio),width:1.4,dash:'4,3'}],
      yFormat:v=>v.toFixed(0)}));
    charts.push(new CE.TimeSeriesChart(document.getElementById('chart-cond4'), {
      title:t('c4_title'), subtitle:t('c4_sub', p.ema_short, p.ema_long), dates:S.dates, height:230,
      series:[{label:'McClellan',color:'#E0904A',data:S.mcclellan,type:'bar'}],
      hlines:[{value:0,color:'#9AA2B5',label:t('zero_axis')}], yFormat:v=>v.toFixed(0)}));

    document.getElementById('panel-sub').textContent = t('panel_sub',
      pct(m.cond_rate.cond1,1), pct(m.cond_rate.cond2,1), pct(m.cond_rate.cond3,1), pct(m.cond_rate.cond4,1),
      m.signal_days, m.n_days.toLocaleString());

    document.getElementById('history-ddhead').textContent = t('dd_head', gw);
    document.getElementById('history-sub').textContent = t('history_sub', gw, p.dd_mode === 'peak_trough');
    document.getElementById('history-tbody').innerHTML = d.clusters.map(c => {
      const dd = c['dd'+gw];
      return `<tr><td class="num">${c.id}</td><td class="strong">${c.start} ~ ${c.end}</td>
        <td class="num">${c.count}</td>
        <td><span class="tag ${c.type==='有效集群'?'tag-amber':'tag-blue'}">${v(c.type)}</span></td>
        <td class="num">${dd===null?t('pending_dd'):pct(dd,1)}</td>
        <td><span class="tag ${TAG[c.level]||'tag-blue'}">${v(c.level)}</span></td></tr>`;
    }).join('');

    // 箱线图直接用 Python 算好的分位数，切换策略时不必重算基准率
    document.getElementById('stats-sub').textContent = t('stats_sub', m.n_gradable);
    const groups = p.dd_windows.filter(w=>m.cluster_box[w])
      .map(w => ({label:t('fwd', w), ...m.cluster_box[w], color:'#D9A441'}));
    if (groups.length) charts.push(new CE.BoxPlotChart(document.getElementById('chart-box'), {
      title:t('box_title'), subtitle:t('box_sub', m.n_gradable),
      groups, yFormat:v=>(v*100).toFixed(0)+'%'}));

  }

  document.querySelectorAll('.toggle-btn').forEach(b =>
    b.addEventListener('click', () => render(b.dataset.cfg)));
  render(Object.keys(D)[0]);

  if (window.__TIMELINE__){
    const TL = window.__TIMELINE__;
    const lanes = TL.lanes.map(lane => ({...lane, label: v(lane.label),
      items: lane.items.map(it => ({...it,
        label: `${v(it.strategy)} #${it.id} · ${v(it.level)}`,
        sublabel: t('mk_range', it.start, it.end, it.count)}))}));
    new CE.TimelineCompare(document.getElementById('chart-timeline'), {
      title:t('tl_title'), subtitle:t('tl_sub'),
      dates:TL.dates, lanes, links:TL.links});
  }
})();
"""

In [6]:
# 参数实验室专用样式
LAB_CSS = """
.lab-layout{display:grid; grid-template-columns:300px 1fr; gap:20px; align-items:start;}
@media (max-width:900px){ .lab-layout{grid-template-columns:1fr;} }
.lab-panel{background:var(--card); border:1px solid var(--line); border-radius:12px;
  padding:18px 18px 8px; position:sticky; top:60px;}
.lab-panel h3{font-family:var(--font-display); font-size:15px; margin:0 0 4px;}
.lab-panel .hint{font-size:11.5px; color:var(--text-faint); margin:0 0 16px; line-height:1.5;}
.slider-row{margin-bottom:16px;}
.slider-row label{display:flex; justify-content:space-between; align-items:baseline;
  font-size:12.5px; color:var(--text-dim); margin-bottom:6px;}
.slider-row label b{font-family:var(--font-mono); color:var(--amber); font-size:13px;}
.slider-row input[type=range]{width:100%; accent-color:var(--amber); cursor:pointer;}
.slider-row .base{font-size:10.5px; color:var(--text-faint); font-family:var(--font-mono);}
.lab-actions{display:flex; gap:8px; margin:6px 0 14px;}
.lab-actions button{flex:1; font-family:var(--font-body); font-size:12.5px; padding:8px;
  border-radius:7px; border:1px solid var(--line); background:var(--panel);
  color:var(--text-dim); cursor:pointer; transition:.15s;}
.lab-actions button:hover{color:var(--amber); border-color:var(--amber);}
.heavy-box{background:rgba(92,138,230,.07); border:1px solid rgba(92,138,230,.3);
  border-radius:9px; padding:12px 14px; margin-top:6px;}
.heavy-box h4{font-size:12.5px; margin:0 0 8px; color:var(--blue);}
.heavy-box .kv{display:flex; justify-content:space-between; font-size:11.5px;
  font-family:var(--font-mono); color:var(--text-faint); padding:2px 0;}
.heavy-box .note{font-size:11px; color:var(--text-faint); margin-top:8px; line-height:1.5;}
.delta{font-family:var(--font-mono); font-size:12px; margin-left:6px;}
.delta.up{color:var(--teal);} .delta.down{color:var(--red);} .delta.flat{color:var(--text-faint);}
.lab-stat{display:grid; grid-template-columns:repeat(auto-fit,minmax(150px,1fr)); gap:12px; margin-bottom:18px;}
.lab-stat .cell{background:var(--card); border:1px solid var(--line); border-radius:10px; padding:14px 16px;}
.lab-stat .cell .lab{font-size:11.5px; color:var(--text-faint); margin-bottom:6px;}
.lab-stat .cell .val{font-family:var(--font-mono); font-size:22px; font-weight:600; color:var(--amber); line-height:1;}
"""


def build_lab(payloads: dict, out_path: Path, lang: str = 'zh') -> Path:
    """payloads: {键: 导出的 JSON}，按 lang 写出参数实验室页面。"""
    btns = toggle_buttons(payloads)
    sel_style = ('width:100%;padding:6px;border-radius:6px;background:var(--panel);'
                 'color:var(--text);border:1px solid var(--line);font-size:12.5px')

    html = _head(TITLES['lab'][lang], LAB_CSS, lang) + f"""
<nav class="topnav">
  <span class="brand">HINDENBURG·LAB</span>
  <a href="{page_name('dashboard', lang)}" data-i18n="lab_back"></a>
  {lang_link('lab', lang)}
</nav>

<header class="hero" style="padding-bottom:24px">
  <div class="eyebrow"><span class="beacon"></span><span data-i18n="lab_eyebrow"></span></div>
  <h1 data-i18n="hero_lab"></h1>
  <p class="sub" data-i18n="lab_intro"></p>
</header>

<div class="section" style="padding-bottom:0">
  <div class="toggle-row">
    <div class="toggle-group">{btns}</div>
    <span class="toggle-caption" id="lab-caption"></span>
  </div>
</div>

<div class="section">
<div class="lab-layout">
  <div class="lab-panel">
    <h3 data-i18n="light_params"></h3>
    <p class="hint" data-i18n="light_hint"></p>
    <div id="sliders"></div>
    <div class="slider-row">
      <label><span data-i18n="cluster_mode"></span><b id="val-cm"></b></label>
      <select id="sel-cm" style="{sel_style}">
        <option value="fixed" data-i18n="opt_fixed"></option>
        <option value="chain" data-i18n="opt_chain"></option>
      </select>
    </div>
    <div class="slider-row">
      <label><span data-i18n="dd_mode"></span><b id="val-dm"></b></label>
      <select id="sel-dm" style="{sel_style}">
        <option value="peak_trough" data-i18n="opt_peak"></option>
        <option value="from_signal" data-i18n="opt_from"></option>
      </select>
    </div>
    <div class="lab-actions">
      <button id="btn-reset" type="button" data-i18n="reset"></button>
    </div>
    <div class="heavy-box">
      <h4 data-i18n="heavy_title"></h4>
      <div id="heavy-list"></div>
      <p class="note" data-i18n="heavy_note"></p>
    </div>
  </div>

  <div>
    <div class="lab-stat" id="lab-stat"></div>
    <div class="section-note warn" id="lab-note" style="margin-bottom:18px"></div>
    <div id="lab-trend"></div>
    <div style="height:16px"></div>
    <div class="chart-grid cols-2-2">
      <div id="lab-cond1"></div><div id="lab-cond2"></div>
      <div id="lab-cond3"></div><div id="lab-cond4"></div>
    </div>
    <div style="height:16px"></div>
    <div id="lab-box"></div>
    <div style="height:16px"></div>
    <div class="tbl-wrap"><div class="tbl-scroll"><table class="rpt">
      <thead><tr><th data-i18n="col_cluster"></th><th data-i18n="col_period"></th><th data-i18n="col_count"></th><th data-i18n="col_type"></th><th data-i18n="col_drop"></th><th data-i18n="col_grade"></th></tr></thead>
      <tbody id="lab-tbody"></tbody></table></div></div>
  </div>
</div>
</div>

<footer>
  <p data-i18n="lab_foot1"></p>
  <p data-i18n="lab_foot2"></p>
</footer>
""" + _scripts(lang, f"""window.__DATA__ = {json.dumps(payloads, ensure_ascii=False)};
window.__SLIDERS__ = {json.dumps(SLIDER_SPECS, ensure_ascii=False)};
window.__LEVEL_COLOR__ = {json.dumps(LEVEL_COLOR, ensure_ascii=False)};""", LAB_JS)

    out_path.write_text(html, encoding='utf-8')
    return out_path

In [7]:
LAB_JS = r"""
(function(){
  const D = window.__DATA__, SPECS = window.__SLIDERS__, LC = window.__LEVEL_COLOR__;
  const {TAG, pct} = window.HO;
  const {t, v} = window.I18N;
  let key = Object.keys(D)[0];
  let params = null, baseline = null, charts = [];

  function defaults(){ return JSON.parse(JSON.stringify(D[key].params)); }

  // 滑块数值显示：百分比或「数值 + 单位」
  const fmt = (s, x) => s.pct ? (x*100).toFixed(1)+'%' : x + t('u_'+s.key);

  function buildSliders(){
    const box = document.getElementById('sliders');
    box.innerHTML = SPECS.map(s => {
      const base = D[key].params[s.key];
      const val = params[s.key];
      return `<div class="slider-row">
        <label>${t('sl_'+s.key)}<b id="val-${s.key}">${fmt(s, val)}</b></label>
        <input type="range" id="sl-${s.key}" min="${s.min}" max="${s.max}" step="${s.step}" value="${val}">
        <div class="base">${t('sl_default', fmt(s, base))}</div></div>`;
    }).join('');
    ['cm','dm'].forEach(k => {
      const key = k === 'cm' ? 'cluster_mode' : 'dd_mode';
      const sel = document.getElementById('sel-'+k);
      sel.value = params[key] || (k==='cm'?'fixed':'peak_trough');
      document.getElementById('val-'+k).textContent = sel.value;
      sel.onchange = e => {
        params[key] = e.target.value;
        document.getElementById('val-'+k).textContent = e.target.value;
        recompute();
      };
    });
    SPECS.forEach(s => {
      document.getElementById('sl-'+s.key).addEventListener('input', e => {
        params[s.key] = parseFloat(e.target.value);
        document.getElementById('val-'+s.key).textContent = fmt(s, params[s.key]);
        recompute();
      });
    });
    // 重参数只读展示
    const heavy = D[key].heavy_keys || [];
    document.getElementById('heavy-list').innerHTML = heavy.map(k =>
      `<div class="kv"><span>${k}</span><span>${D[key].params[k]}</span></div>`).join('')
      + `<div class="kv"><span>benchmark</span><span>${D[key].benchmark}</span></div>`
      + `<div class="kv"><span>state_rule</span><span>${v(D[key].state_rule)}</span></div>`;
  }

  function delta(cur, base, invert){
    if (cur===null||base===null||cur===undefined||base===undefined) return '';
    const d = cur - base;
    if (Math.abs(d) < 1e-9) return '<span class="delta flat">±0</span>';
    const good = invert ? d < 0 : d > 0;
    const sign = d > 0 ? '+' : '';
    const txt = Math.abs(d) < 1 ? sign+(d*100).toFixed(1)+'pp' : sign+d;
    return `<span class="delta ${good?'up':'down'}">${txt}</span>`;
  }

  function recompute(){
    const d = D[key], S = d.series, src = v(d.index_source);
    const r = BT.runAll(S, params, D[key].index_pre);
    const m = r.meta;
    const bm = baseline.meta;
    const gw = params.grade_window, cuts = params.grade_cuts;

    const baseGW = m.baseline_all ? m.baseline_all[gw] : null;
    const exVal = m.excess ? m.excess.vs_all : null;
    document.getElementById('lab-stat').innerHTML = `
      <div class="cell"><div class="lab">${t('ls_signal_days')}</div><div class="val">${m.signal_days}${delta(m.signal_days, bm.signal_days)}</div></div>
      <div class="cell"><div class="lab">${t('ls_valid')}</div><div class="val">${m.n_clusters_valid}${delta(m.n_clusters_valid, bm.n_clusters_valid)}</div></div>
      <div class="cell"><div class="lab">${t('ls_gradable')}</div><div class="val">${m.n_gradable}${delta(m.n_gradable, bm.n_gradable)}</div></div>
      <div class="cell"><div class="lab">${t('ls_hit')}</div><div class="val">${pct(m.hit_rate,0)}${delta(m.hit_rate, bm.hit_rate)}</div></div>
      <div class="cell"><div class="lab">${t('ls_hit_mid', gw, (cuts[1]*100).toFixed(0))}</div><div class="val">${m.hit_mid}${delta(m.hit_mid, bm.hit_mid)}</div></div>
      <div class="cell"><div class="lab">${t('ls_hit_big', (cuts[0]*100).toFixed(0))}</div><div class="val">${m.hit_big}${delta(m.hit_big, bm.hit_big)}</div></div>
      <div class="cell"><div class="lab">${t('ls_base')}</div><div class="val">${baseGW?pct(baseGW.p_mid,0):'—'}</div></div>
      <div class="cell"><div class="lab">${t('ls_excess')}</div><div class="val">${exVal===null?'—':(exVal>=0?'+':'')+(exVal*100).toFixed(0)+'pp'}${delta(exVal, bm.excess?bm.excess.vs_all:null)}</div></div>`;

    document.getElementById('lab-note').innerHTML = m.n_gradable < 3
      ? t('lab_few', m.n_gradable)
      : t('lab_now', m.n_gradable, pct(m.hit_rate,0), baseGW?pct(baseGW.p_mid,0):'—',
          exVal===null?'—':(exVal>=0?'+':'')+(exVal*100).toFixed(0)+'pp');

    charts.forEach(c => c.destroy && c.destroy());
    charts = [];
    const markers = HO.signalMarkers(r.signalDays, r.clusters, LC);
    const gap = S.index_close.findIndex(v => v !== null);
    const bands = gap>0 ? [{i0:0,i1:gap,color:'#5C8AE6',opacity:0.10,label:t('no_index')}] : [];

    charts.push(new CE.TimeSeriesChart(document.getElementById('lab-trend'), {
      title: t('lab_trend_title', src), subtitle:t('lab_trend_sub'),
      dates:S.dates, height:320,
      series:[{label:src,color:'#D9A441',data:S.index_close,width:1.5}],
      markers, bands, yFormat:v=>v.toFixed(0)}));

    charts.push(new CE.TimeSeriesChart(document.getElementById('lab-cond1'), {
      title:t('lab_c1', params.ma_period), subtitle:t('holds', pct(m.cond_rate.cond1,1)),
      dates:S.dates, height:210,
      series:[{label:t('index'),color:'#97A3BC',data:S.index_close,width:1.1},
              {label:'MA',color:'#D9A441',data:r.ma,width:1.8}],
      bands, yFormat:v=>v.toFixed(0)}));
    charts.push(new CE.TimeSeriesChart(document.getElementById('lab-cond2'), {
      title:t('lab_c2', (params.threshold*100).toFixed(1)), subtitle:t('holds', pct(m.cond_rate.cond2,1)),
      dates:S.dates, height:210,
      series:[{label:t('new_high_pct'),color:'#C64B37',data:S.high_ratio,width:1.3},
              {label:t('new_low_pct'),color:'#5C8AE6',data:S.low_ratio,width:1.3}],
      hlines:[{value:params.threshold,color:'#D9A441',label:t('threshold')}],
      yFormat:v=>(v*100).toFixed(1)+'%'}));
    charts.push(new CE.TimeSeriesChart(document.getElementById('lab-cond3'), {
      title:t('lab_c3', params.hl_ratio.toFixed(1)), subtitle:t('holds', pct(m.cond_rate.cond3,1)),
      dates:S.dates, height:210,
      series:[{label:t('new_high_n'),color:'#C64B37',data:S.new_high,type:'bar',opacity:0.75},
              {label:t('threshold'),color:'#5C8AE6',data:S.new_low.map(v=>v==null?null:v*params.hl_ratio),width:1.4,dash:'4,3'}],
      yFormat:v=>v.toFixed(0)}));
    charts.push(new CE.TimeSeriesChart(document.getElementById('lab-cond4'), {
      title:t('lab_c4'), subtitle:t('lab_c4_sub', pct(m.cond_rate.cond4,1)),
      dates:S.dates, height:210,
      series:[{label:'McClellan',color:'#E0904A',data:S.mcclellan,type:'bar'}],
      hlines:[{value:0,color:'#7d8aa0',label:t('zero_axis')}], yFormat:v=>v.toFixed(0)}));

    const groups = params.dd_windows.filter(w=>r.box[w]).map(w =>
      ({label:t('fwd', w), ...r.box[w], color:'#D9A441'}));
    if (groups.length) charts.push(new CE.BoxPlotChart(document.getElementById('lab-box'), {
      title:t('lab_box_title'), subtitle:t('lab_box_sub', m.n_gradable),
      groups, yFormat:v=>(v*100).toFixed(0)+'%'}));
    else document.getElementById('lab-box').innerHTML = '';

    document.getElementById('lab-tbody').innerHTML = r.clusters.length ? r.clusters.map(c => {
      const dd = c['dd'+gw];
      return `<tr><td class="num">${c.id}</td><td class="strong">${c.start} ~ ${c.end}</td>
        <td class="num">${c.count}</td>
        <td><span class="tag ${c.type==='有效集群'?'tag-amber':'tag-blue'}">${v(c.type)}</span></td>
        <td class="num">${dd===null||dd===undefined?t('pending_dd'):pct(dd,1)}</td>
        <td><span class="tag ${TAG[c.level]||'tag-blue'}">${v(c.level)}</span></td></tr>`;
    }).join('') : `<tr><td colspan="6" style="text-align:center;color:var(--text-faint)">${t('lab_no_signal')}</td></tr>`;
  }

  function switchTo(k){
    key = k;
    params = defaults();
    baseline = BT.runAll(D[key].series, D[key].params, D[key].index_pre);
    document.querySelectorAll('.toggle-btn').forEach(b => b.classList.toggle('active', b.dataset.cfg===k));
    document.getElementById('lab-caption').textContent =
      `${v(D[key].strategy)} · ${v(D[key].index_source)} · ${v(D[key].state_rule)}`;
    buildSliders();
    recompute();
  }

  document.querySelectorAll('.toggle-btn').forEach(b =>
    b.addEventListener('click', () => switchTo(b.dataset.cfg)));
  document.getElementById('btn-reset').addEventListener('click', () => {
    params = defaults(); buildSliders(); recompute();
  });

  switchTo(key);
})();
"""

## 3. 生成网页

In [8]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
dashboard_payloads = {k: PAYLOADS[k] for k in DASHBOARD_ORDER if k in PAYLOADS}
for lang in LANGS:
    for path in (build_dashboard(dashboard_payloads, OUT_DIR / page_name('dashboard', lang), lang),
                 build_lab(PAYLOADS, OUT_DIR / page_name('lab', lang), lang)):
        print(f'已生成 {path}（{path.stat().st_size / 1024:,.0f} KB）')

已生成 docs/dashboard.html（1,394 KB）
已生成 docs/lab.html（1,551 KB）


已生成 docs/dashboard_en.html（1,394 KB）
已生成 docs/lab_en.html（1,551 KB）
